<a href="https://colab.research.google.com/github/AmnaNoor123/urdu-ocr-codesaviours-si26-amna/blob/main/SI26_Week2_amna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**This notebook covers Week 2:**

Image Preprocessing for the Urdu OCR project.

The images we collected vary in size, color, and pixel quality. Before training any model, we need to standardize them — this is called preprocessing. It ensures the model can learn smoothly from consistent, clean input rather than noisy, inconsistent raw data.

In this notebook, we will:

Convert images to grayscale
Resize all images to a standard dimension
Remove noise for cleaner text
Binarize the images (pure black/white pixels)
Test Tesseract OCR on the processed Urdu images to see where it fails

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/Books.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('data/raw')

print('Unzipped! these folders:')
print(os.listdir('data/raw'))

Unzipped! these folders:
['Sign boards', 'Books', 'Newspaper', 'Other', 'Synthetic raw images']


In [5]:
!pip install opencv-python-headless pillow
import cv2
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt
print('Libraries loaded successfully!')

Libraries loaded successfully!


In [6]:
def preprocess_image(image_path, save_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f'Could not load: {image_path}')
        return
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (512, 128))
    denoised = cv2.fastNlMeansDenoising(resized, h=10)
    _, binary = cv2.threshold(denoised, 127, 255, cv2.THRESH_BINARY)
    cv2.imwrite(save_path, binary)
    return binary

os.makedirs('data/processed', exist_ok=True)
print('Preprocessing function ready!')

Preprocessing function ready!


In [7]:
import glob

all_images = glob.glob('data/raw/**/*.jpg', recursive=True)
all_images += glob.glob('data/raw/**/*.png', recursive=True)
all_images += glob.glob('data/raw/**/*.jpeg', recursive=True)

print(f'Found {len(all_images)} images to process')

processed_count = 0
for img_path in all_images:
    filename = os.path.basename(img_path)
    save_path = f'data/processed/{filename}'
    result = preprocess_image(img_path, save_path)
    if result is not None:
        processed_count += 1

print(f'Done! Processed {processed_count} images')
print('Check data/processed/ folder')

Found 132 images to process
Done! Processed 132 images
Check data/processed/ folder


**Tesseract OCR Test**

In [8]:
!apt-get install -y tesseract-ocr tesseract-ocr-urd
!pip install pytesseract

import pytesseract
from PIL import Image

# 5 processed images pe test karo
test_images = list(glob.glob('data/processed/*.png'))[:5]

print('=== Tesseract Results on Urdu Images ===')
print()

for img_path in test_images:
    img = Image.open(img_path)
    result = pytesseract.image_to_string(img, lang='urd')
    print(f'Image: {img_path}')
    print(f'Tesseract output: {result}')
    print('---')

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-urd
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 1,000 kB of archives.
After this operation, 1,413 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-urd all 1:4.00~git30-7274cfa-1.1 [1,000 kB]
Fetched 1,000 kB in 1s (904 kB/s)
Selecting previously unselected package tesseract-ocr-urd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-urd_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
=== Tesseract Results on Urdu Images ===

Image: data/processed/urdu_2 (1).png
Tesseract output: آج کا موم خوشوارے

---
Im

**Gap Analysis — Tesseract on Urdu Images**

**Image 1: urdu_2.png**
- Actual text: آج کا موسم خوشگوار ہے
- Tesseract output: آج کا موم خوشوارے
- What went wrong: Words got merged/mispelled — "موسم" became "موم" (letters dropped),
  and "خوشگوار ہے" became "خوشوارے" (multiple letters missing, words incorrectly joined).

**Image 2: ذ_98.png**
- Actual text: ذ
- Tesseract output: 0
- What went wrong: A single Urdu letter was misread entirely as the number "0" —
  complete character misclassification.

**Image 3: س_97.png**
- Actual text: س
- Tesseract output: (blank)
- What went wrong: Tesseract failed to detect any text at all — isolated Urdu
  characters without surrounding context are not recognized.

**Image 4: ا_974.png**
- Actual text: ا
- Tesseract output: (blank)
- What went wrong: Same as above — no text detected.

**Image 5: ا_973.png**
- Actual text: ا
- Tesseract output: (blank)
- What went wrong: Same as above — no text detected.

**Summary:**
Tesseract fails on Urdu because it struggles with the connected, cursive nature
of Urdu script — letters change shape depending on their position in a word, and
Tesseract's model isn't well-trained to handle this joining behavior. It performs
worst on single isolated characters (returning blank or wrong symbols entirely),
and even on full sentences it drops or merges letters, producing text that looks
similar to the original but is factually incorrect. This confirms the need for a
custom-trained OCR model specifically built for Urdu's script structure.